In [ ]:
# Name: Mudit Airan
# Description: Home Security Monitoring System
# A small program that simulates a home security system: initializes sensors, checks readings against thresholds, raises alerts depending on the current system mode

import datetime
import random
from dataclasses import dataclass, field
from typing import Optional

def create_sensor(sensor_id, location, sensor_type, value):
    return {
        "id": sensor_id,
        "location": location,
        "type": sensor_type,       # "temperature", "motion", "door", "smoke"
        "current_value": value,
    }


def create_alert(level, message, sensor_id, timestamp):
    return {
        "level": level,            # "SECURITY", "SAFETY", or "COMFORT"
        "message": message,
        "sensor_id": sensor_id,
        "timestamp": timestamp,
    }

def initialize_sensors():
    return [
        create_sensor("TEMP-01", "Living Room", "temperature", 70.0),
        create_sensor("MOTION-01", "Hallway", "motion", False),
        create_sensor("DOOR-01", "Front Door", "door", "closed"),
        create_sensor("SMOKE-01", "Kitchen", "smoke", False),
    ]

sensors = initialize_sensors()
for s in sensors:
    print(f"Created {s['id']} ({s['type']}) in {s['location']}")


def check_temperature(value, mode):
    """Return an alert message string, or None if everything is fine."""
    if value < 35:
        return ("SAFETY", f"Temperature {value}°F - frozen pipe risk!")
    elif value > 95:
        return ("SAFETY", f"Temperature {value}°F - equipment failure risk!")
    elif mode == "home" and not (65 <= value <= 75):
        return ("COMFORT", f"Temperature {value}°F outside comfort range (65-75°F).")
    else:
        return None
    

def check_motion(detected, mode):
    if detected and mode == "away":
        return ("SECURITY", "Motion detected while house is in AWAY mode!")
    return None

def check_door(state, mode):
    if state == "open" and mode == "away":
        return ("SECURITY", "Door opened while house is in AWAY mode!")
    return None

def check_smoke(detected, mode):
    if detected:                      # smoke is always bad, mode doesn't matter
        return ("SAFETY", "SMOKE detected - fire risk!")
    return None


# Temperature: each branch of the if/elif chain
print(check_temperature(30.0, "home"))   # expect ('SAFETY', ...freeze...)
print(check_temperature(100.0, "away"))  # expect ('SAFETY', ...equipment...)
print(check_temperature(60.0, "home"))   # expect ('COMFORT', ...)
print(check_temperature(60.0, "away"))   # expect None  <- comfort only matters at home
print(check_temperature(70.0, "home"))   # expect None

# Mode logic: same reading, different mode
print(check_motion(True, "away"))        # expect ('SECURITY', ...)
print(check_motion(True, "home"))        # expect None
print(check_smoke(True, "home"))         # expect ('SAFETY', ...) regardless of mode

def generate_reading(sensor):
    if sensor["type"] == "temperature":
        # nudge the temperature up or down a tiny bit
        change = random.uniform(-1.5, 1.5)
        sensor["current_value"] = sensor["current_value"] + change

    elif sensor["type"] == "motion":
        sensor["current_value"] = random.choice([True, False])

    elif sensor["type"] == "door":
        sensor["current_value"] = random.choice(["open", "closed"])

    elif sensor["type"] == "smoke":
        # 4 Falses and 1 True in the list = smoke happens 1 time in 5
        sensor["current_value"] = random.choice([False, False, False, False, True])


s = create_sensor("TEMP-01", "Living Room", "temperature", 70.0)
generate_reading(s)
print(s["current_value"]) 

def process_reading(sensor, mode):
    if sensor["type"] == "temperature":
        return check_temperature(sensor["current_value"], mode)
    elif sensor["type"] == "motion":
        return check_motion(sensor["current_value"], mode)
    elif sensor["type"] == "door":
        return check_door(sensor["current_value"], mode)
    elif sensor["type"] == "smoke":
        return check_smoke(sensor["current_value"], mode)
    
def trigger_alert(level, message, alert_list):
    print("*** ALERT [" + level + "]: " + message + " ***")
    alert_list.append(message)


from datetime import datetime

def log_event(event_log, message):
    time_now = datetime.now().strftime("%H:%M:%S")     # e.g. "14:32:07"
    entry = "[" + time_now + "] " + message
    event_log.append(entry)
    print(entry)

# Empty lists to collect things in
alerts = []
log = []

# A sensor we KNOW is a problem: 30°F is below the 35°F threshold
test_sensor = create_sensor("TEMP-01", "Basement", "temperature", 30.0)

# Ask: is this a problem?
result = process_reading(test_sensor, "home")
print(result)     # should show: ('SAFETY', 'Temperature 30.0°F - frozen pipe risk!')

# If a problem came back, sound the alarm and write it in the diary
if result is not None:
    level = result[0]      # first item of the pair: "SAFETY"
    message = result[1]    # second item: the description
    trigger_alert(level, message, alerts)
    log_event(log, "Alert from sensor " + test_sensor["id"])

print("Number of alerts:", len(alerts))    # should say 1
print("Number of log entries:", len(log))  # should say 1


d = create_sensor("DOOR-01", "Front Door", "door", "closed")
generate_reading(d)
print(d["current_value"])
generate_reading(d)
print(d["current_value"])
generate_reading(d)
print(d["current_value"])

In [ ]:
import random
from statistics import mode

class Sensor:
    def __init__(self, sensor_id, location, sensor_type, value):
        # store the facts inside this sensor
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.current_value = value
        self.starting_value = value      # remember this for reset() later

    def read(self):
        # give this sensor a new pretend value (was generate_reading)
        if self.type == "temperature":
            change = random.uniform(-1.5, 1.5)
            self.current_value = self.current_value + change

        elif self.type == "motion":
            self.current_value = random.choice([True, False])

        elif self.type == "door":
            self.current_value = random.choice(["open", "closed"])

        elif self.type == "smoke":
            self.current_value = random.choice([False, False, False, False, True])

    def isAbnormal(self, mode):
        # is this reading a problem? (was your check_* functions)
        # answer: a (level, message) pair, or None if all is fine
        if self.type == "temperature":
            if self.current_value < 35:
                return ("SAFETY", f"Temperature {self.current_value:.1f}°F - frozen pipe risk!")
            elif self.current_value > 95:
                return ("SAFETY", f"Temperature {self.current_value:.1f}°F - equipment failure risk!")
            elif mode == "home" and not (65 <= self.current_value <= 75):
                return ("COMFORT", f"Temperature {self.current_value:.1f}°F outside comfort range.")

        elif self.type == "motion":
            if self.current_value and mode == "away":
                return ("SECURITY", "Motion detected while house is in AWAY mode!")

        elif self.type == "door":
            if self.current_value == "open" and mode == "away":
                return ("SECURITY", "Door opened while house is in AWAY mode!")

        elif self.type == "smoke":
            if self.current_value:
                return ("SAFETY", "SMOKE detected - fire risk!")

        return None      # nothing matched = everything is normal

    def reset(self):
        # put the sensor back to the value it started with
        self.current_value = self.starting_value

    def describe(self):
        # returns a friendly text version of the current reading
        if self.type == "temperature":
            if 65 <= self.current_value <= 75:
                status = "Normal"
            else:
                status = "Check"
            return f"{self.location} Temperature: {self.current_value:.0f}°F ({status})"

        elif self.type == "motion":
            if self.current_value:
                return self.location + " Motion: ACTIVITY DETECTED"
            else:
                return self.location + " Motion: No activity"

        elif self.type == "door":
            if self.current_value == "open" and mode in ["away", "sleep"]:
                return self.location + ": OPENED"
            else:
                return self.location + ": CLOSED"

        elif self.type == "smoke":
            if self.current_value:
                return self.location + " Smoke: DETECTED"
            else:
                return self.location + " Smoke: CLEAR"

"""
# Build one sensor from the blueprint
temp = Sensor("TEMP-01", "Living Room", "temperature", 70.0)

# Check its facts
print(temp.id)             # TEMP-01
print(temp.location)       # Living Room
print(temp.current_value)  # 70.0

# Ask it to read a new value a few times
temp.read()
print(temp.current_value)  # near 70, like 70.9
temp.read()
print(temp.current_value)  # drifted again

# Ask it to check itself (70-ish is fine, expect None)
print(temp.isAbnormal("home"))    # None

# Force a bad value and check again
temp.current_value = 30.0
print(temp.isAbnormal("home"))    # ('SAFETY', 'Temperature 30.0°F - frozen pipe risk!')

# Reset back to the beginning
temp.reset()
print(temp.current_value)         # 70.0 again
"""

def initialize_sensors():
    return [
        Sensor("TEMP-01", "Living Room", "temperature", 70.0),
        Sensor("MOTION-01", "Hallway", "motion", False),
        Sensor("DOOR-01", "Front Door", "door", "closed"),
        Sensor("SMOKE-01", "Kitchen", "smoke", False),
    ]

import time
from datetime import datetime


def trigger_alert(level, message, alerts, log):
    # SECURITY and SAFETY are serious; COMFORT is minor
    if level == "COMFORT":
        priority = "LOW"
    else:
        priority = "HIGH"

    print("[ALERT!] 🚨 " + priority + ": " + level + ": " + message)
    alerts.append(message)

    time_now = datetime.now().strftime("%H:%M:%S")
    entry = "[LOG] [" + time_now + "] Sending notification to homeowner..."
    print(entry)
    log.append(entry)

"""
print("\n########## TEST 4: SAFETY (forced) ##########")
alerts = []
log = []

# Force a freezing sensor — no randomness involved
frozen = Sensor("TEMP-99", "Basement", "temperature", 30.0)
print("[READING] " + frozen.describe())
result = frozen.isAbnormal("home")
if result is not None:
    trigger_alert(result[0], result[1], alerts, log)
# EXPECT: 🚨 HIGH: SAFETY: frozen pipe alert

# Force a smoking sensor
smoky = Sensor("SMOKE-99", "Kitchen", "smoke", True)
print("[READING] " + smoky.describe())
result = smoky.isAbnormal("home")
if result is not None:
    trigger_alert(result[0], result[1], alerts, log)
# EXPECT: 🚨 HIGH: SAFETY: smoke alert

# Force an overheating sensor
hot = Sensor("TEMP-98", "Garage", "temperature", 100.0)
print("[READING] " + hot.describe())
result = hot.isAbnormal("home")
if result is not None:
    trigger_alert(result[0], result[1], alerts, log)
# EXPECT: 🚨 HIGH: SAFETY: equipment failure alert
""" 


def run_simulation(cycles, mode):
    # ---- SETUP ----
    sensors = initialize_sensors()
    alerts = []
    log = []

    print("=== HomeGuard Security System ===")
    print("Mode: " + mode.upper())
    # ---- THE HEARTBEAT ----
    for cycle in range(1, cycles + 1):
        print()
        time_now = datetime.now().strftime("%H:%M:%S")
        print("Time: " + time_now)

        for sensor in sensors:
            sensor.read()                            # 1. new value
            print("[READING] " + sensor.describe())  # 2. show it (NEW!)

            result = sensor.isAbnormal(mode)         # 3. problem?
            if result is not None:                   # 4. if yes, alarm
                trigger_alert(result[0], result[1], alerts, log)

        time.sleep(1)

    # ---- SUMMARY ----
    print()
    print("=== Simulation complete ===")
    print("Cycles: " + str(cycles) + " | Mode: " + mode.upper()
          + " | Alerts: " + str(len(alerts)))


run_simulation(3, "home")
